# Challenge 02 — News Agent (Code-First)

In this challenge, you'll build a **News Agent** programmatically using the **Azure AI Foundry Agent SDK v2** and Python. By the end, you'll be able to chat with your agent from this notebook.

## Learning objectives
- Authenticate to Azure AI Foundry from code
- Create an agent with custom instructions using AIProjectClient
- Send messages and process agent runs
- Inspect agent responses

## ⚙️ Kernel Setup

Before running this notebook, you need to select the correct Python kernel pointing to the virtual environment created in `Student/Resources/`:

1. Open the **Command Palette** (`Ctrl+Shift+P`)
2. Type **"Notebook: Select Notebook Kernel"** and select it
3. Choose **"Python Environments…"**
4. Select the `.venv` interpreter located at:
   ```
   xxx-FoundryAgents/Student/Resources/.venv/Scripts/python.exe
   ```
   (If you don't see it, click **"Enter interpreter path…"** and browse to that path)

> **Tip:** If you haven't created the venv yet, run the pip install cell below first from a terminal with your cwd set to `Student/Resources/`, then restart VS Code and pick the kernel.

Set up your Python environment for the notebook

In [ ]:
# From inside the WhatTheHack folder
python -m venv .venv

# Activate it 
source .venv/bin/activate


# Upgrade pip
python -m pip install --upgrade pip

# Install the core packages you'll need for the News Agent challenge
pip install azure-ai-projects azure-identity ipykernel python-dotenv requests

 Imports & env setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    PromptAgentDefinition,
)

# Load env vars from Student/Resources/.env
env_path = Path(__file__).resolve().parent.parent.parent / "Student" / "Resources" / ".env" if "__file__" in dir() else Path.cwd().parent.parent / "Student" / "Resources" / ".env"
load_dotenv(dotenv_path=env_path, override=True)

PROJECT_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")  # optional but recommended

assert PROJECT_ENDPOINT, "AZURE_AI_FOUNDRY_ENDPOINT is missing from .env"
assert MODEL_DEPLOYMENT_NAME, "AZURE_OPENAI_DEPLOYMENT_NAME is missing from .env"

print(f"Endpoint: {PROJECT_ENDPOINT}")
print(f"Model: {MODEL_DEPLOYMENT_NAME}")
print(f"Tenant: {TENANT_ID or '(not set — will use CLI default)'}")

Cell 2 — Create the AIProjectClient


In [ ]:
# Connect to your Foundry project.
# Try Azure CLI first; fall back to a browser sign-in if the CLI session isn't visible to the kernel.
from azure.identity import DefaultAzureCredential


credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

project_client = AIProjectClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)
print("✅ Connected to Azure AI Foundry")

Create the News Agent

In [ ]:
# Define the agent's personality + job
news_agent = project_client.agents.create_version(

    agent_name="news-travel-agent",
    definition=PromptAgentDefinition(
        model=MODEL_DEPLOYMENT_NAME,
        instructions=(
            "You are a helpful travel news assistant. "
            "When a user asks about a destination, summarize the latest news, "
            "events, and travel advisories for that location in a friendly, concise way. "
            "Always end with a fun tip about visiting that place."
        )
    )
)
print(f"✅ Agent created — ID: {news_agent.id}")

Create a conversation + send a message

In [ ]:
# Get an OpenAI client from the project client
openai_client = project_client.get_openai_client()

# Create a conversation with an initial user message
conversation = openai_client.conversations.create(
    items=[{"type": "message", "role": "user", "content": "What's happening in Texas this week that a traveler should know?"}],
)
print(f"✅ Conversation created — ID: {conversation.id}")

Run the agent + see the response

In [ ]:
# Run the agent against the conversation
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body={"agent_reference": {"name": news_agent.name, "type": "agent_reference"}},
)
print(f"✅ Agent responded")
print(f"\n--- AGENT ---\n{response.output_text}")

 Cleanup


In [ ]:
# Delete the conversation and agent to keep your Foundry project clean
openai_client.conversations.delete(conversation_id=conversation.id)
print(f"✅ Conversation deleted — ID: {conversation.id}")

project_client.agents.delete_version(agent_name=news_agent.name, agent_version=news_agent.version)
print(f"✅ Agent deleted — name: {news_agent.name}, version: {news_agent.version}")